# Fine-Tune GPT-OSS-20B — SAP Master-Data Cleaning

Runs on a free Google Colab **T4** (~15 min for 30 steps). LoRA (0.6% params) on `unsloth/gpt-oss-20b`, export to GGUF for local inference.

*Input data:* `data/train.jsonl` (messy → clean pairs) generated by `scripts/gen_data.py`.


In [ ]:
# Cell: install deps (Colab)
!pip install -q unsloth
!pip install -q --no-deps trl==0.22.2 torch transformers==4.56.2 datasets


In [ ]:
# Cell: imports & model loading
import json
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel, is_bfloat16_supported

MODEL_NAME = "unsloth/gpt-oss-20b"
MAX_SEQ_LEN = 1024
DTYPE = None  # auto bfloat16/float16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=DTYPE,
    load_in_4bit=True,
)


In [ ]:
# Cell: attach LoRA adapters (PEFT)
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


In [ ]:
# Cell: formatter -> chat template (system/user/assistant)
def formatting_prompts_func(examples):
    outs = []
    for messy, clean in zip(examples["messy"], examples["clean"]):
        messages = [
            {"role": "system", "content":
             "You clean messy SAP-style master data into valid JSON. "
             "Return only the JSON object, no commentary."},
            {"role": "user", "content":
             "Clean this record: " + json.dumps(messy, ensure_ascii=False)},
            {"role": "assistant", "content":
             json.dumps(clean, ensure_ascii=False)},
        ]
        outs.append(tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False))
    return {"text": outs}


In [ ]:
# Cell: load JSONL dataset and prepare response template
dataset = load_dataset(
    "json",
    data_files={"train": "data/train.jsonl", "valid": "data/valid.jsonl"},
)
dataset = dataset.map(formatting_prompts_func, batched=True)

response_template = "<|start_header_id|>assistant<|end_header_id|>\n\n"


In [ ]:
# Cell: SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["valid"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=4,
        max_steps=30,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="output/checkpoints",
        report_to="none",
    ),
)
trainer.train()


In [ ]:
# Cell: save LoRA adapter
model.save_pretrained("output/lora_adapter")
tokenizer.save_pretrained("output/lora_adapter")
print("LoRA saved to output/lora_adapter")


## Fuse + GGUF export (run after training)

Merge the LoRA into the base and export `q8_0` (~600 MB) for local inference:


In [ ]:
# Cell: merge + export GGUF (optional, on T4)
model = FastLanguageModel.from_pretrained(
    model_name="output/lora_adapter", max_seq_length=1024,
    load_in_4bit=False, dtype=torch.float16)
model = model.merge_and_unload()
model.save_pretrained_gguf(
    "output", tokenizer, quantization_method="q8_0",
    model_name="gpt-oss-sap-cleaner",
)
print("GGUF -> output/gpt-oss-sap-cleaner-q8_0.gguf")
